# 🚀 Профессиональное обучение Django с нуля

Полный курс по разработке веб-приложений на Django, от базовых концепций до развертывания на продакшене.

## 1. Установка и настройка окружения

### Шаг 1: Установка Python
- Python 3.8+ рекомендуется
- Проверить: `python --version`

### Шаг 2: Создание виртуального окружения
```bash
# Windows
python -m venv venv
venv\Scripts\activate

# Mac/Linux
python3 -m venv venv
source venv/bin/activate
```

### Шаг 3: Установка Django
```bash
pip install django django-restframework pillow python-dotenv
```

**Best Practice:** Всегда используйте виртуальное окружение для изоляции зависимостей проекта!

## 2. Создание Django проекта и приложения

### Структура проекта
```
myproject/
├── manage.py              # Утилита управления проектом
├── myproject/              # Папка проекта (конфигурация)
│   ├── __init__.py
│   ├── settings.py        # Настройки проекта
│   ├── urls.py            # Главные маршруты
│   ├── asgi.py            # ASGI конфиг
│   └── wsgi.py            # WSGI конфиг
├── app1/                  # Приложение 1
│   ├── migrations/
│   ├── models.py
│   ├── views.py
│   ├── urls.py
│   └── templates/
└── app2/                  # Приложение 2
```

### Команды создания
```bash
# Создать проект
django-admin startproject myproject

# Создать приложение
python manage.py startapp catalog

# Запустить сервер разработки
python manage.py runserver

# Создать суперпользователя
python manage.py createsuperuser
```

**Важно:** Приложение нужно зарегистрировать в `settings.py` в списке `INSTALLED_APPS`

## 3. Модели и работа с БД

### Основные типы полей
```python
from django.db import models

class Movie(models.Model):
    # Текстовые поля
    title = models.CharField(max_length=200, unique=True)
    description = models.TextField(default='')
    slug = models.SlugField(unique=True)
    
    # Числовые поля
    rating = models.IntegerField(default=0)
    duration = models.IntegerField(help_text='В минутах')
    
    # Дата и время
    release_date = models.DateField()
    created_at = models.DateTimeField(auto_now_add=True)
    updated_at = models.DateTimeField(auto_now=True)
    
    # Boolean и выбор
    is_active = models.BooleanField(default=True)
    status = models.CharField(
        max_length=10,
        choices=[
            ('draft', 'Черновик'),
            ('published', 'Опубликовано'),
            ('archived', 'Архив')
        ],
        default='draft'
    )
    
    # Медиа файлы
    poster = models.ImageField(upload_to='posters/')
    
    # Связи между моделями
    category = models.ForeignKey('Category', on_delete=models.SET_NULL, null=True)
    genres = models.ManyToManyField('Genre', related_name='movies')
    
    class Meta:
        ordering = ['-created_at']
        verbose_name = 'Фильм'
        verbose_name_plural = 'Фильмы'
        indexes = [
            models.Index(fields=['slug']),
            models.Index(fields=['-created_at']),
        ]
    
    def __str__(self):
        return self.title
```

### Связи между моделями
- **ForeignKey**: Один-к-множеству (один Film → много Comments)
- **OneToOneField**: Один-к-одному
- **ManyToManyField**: Много-к-много (много Movies → много Genres)

## 4. Представления (Views) и URL маршруты

### Функциональные представления (Function-Based Views)
```python
# views.py
from django.shortcuts import render, get_object_or_404
from django.http import HttpResponse
from .models import Movie

def movie_list(request):
    movies = Movie.objects.filter(is_active=True)
    context = {'movies': movies}
    return render(request, 'movie_list.html', context)

def movie_detail(request, slug):
    movie = get_object_or_404(Movie, slug=slug)
    return render(request, 'movie_detail.html', {'movie': movie})
```

### Классовые представления (Class-Based Views)
```python
from django.views import View
from django.views.generic import ListView, DetailView

class MovieListView(ListView):
    model = Movie
    template_name = 'movie_list.html'
    context_object_name = 'movies'
    paginate_by = 10
    
    def get_queryset(self):
        return Movie.objects.filter(is_active=True)

class MovieDetailView(DetailView):
    model = Movie
    template_name = 'movie_detail.html'
    slug_field = 'slug'
    context_object_name = 'movie'
```

### URL маршруты
```python
# urls.py
from django.urls import path
from . import views

app_name = 'catalog'

urlpatterns = [
    path('movies/', views.MovieListView.as_view(), name='movie_list'),
    path('movies/<slug:slug>/', views.MovieDetailView.as_view(), name='movie_detail'),
]
```

**Best Practice:** Используйте Class-Based Views (CBV) для уменьшения кода, но понимайте и FBV!

## 5. Шаблоны (Templates)

### Структура папок
```
templates/
├── base.html           # Основной шаблон
├── navbar.html         # Меню навигации
├── catalog/
│   ├── movie_list.html
│   └── movie_detail.html
└── auth/
    ├── login.html
    └── register.html
```

### Пример base.html
```html
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    {% load static %}
    <link rel="stylesheet" href="{% static 'css/style.css' %}">
    <title>{% block title %}Cinema{% endblock %}</title>
</head>
<body>
    {% include 'navbar.html' %}
    
    <main class="container">
        {% if messages %}
            {% for message in messages %}
                <div class="alert alert-{{ message.tags }}">
                    {{ message }}
                </div>
            {% endfor %}
        {% endif %}
        
        {% block content %}{% endblock %}
    </main>
    
    {% include 'footer.html' %}
</body>
</html>
```

### Наследование и теги
```html
{% extends 'base.html' %}
{% load static %}

{% block title %}Фильмы{% endblock %}

{% block content %}
    <div class="movies-grid">
        {% for movie in movies %}
            <div class="movie-card">
                <h3>{{ movie.title }}</h3>
                <img src="{{ movie.poster.url }}" alt="{{ movie.title }}">
                <p>{{ movie.description|truncatewords:25 }}</p>
                <a href="{% url 'catalog:movie_detail' movie.slug %}">Подробнее</a>
            </div>
        {% empty %}
            <p>Фильмов не найдено</p>
        {% endfor %}
    </div>
{% endblock %}
```

**Полезные фильтры:** `|truncatewords`, `|length`, `|date:"d M Y"`, `|upper`, `|lower`

## 6. Формы и валидация

### Обычная форма Django
```python
# forms.py
from django import forms
from .models import Movie, Review

class MovieForm(forms.Form):
    title = forms.CharField(
        max_length=200,
        widget=forms.TextInput(attrs={'class': 'form-control'})
    )
    description = forms.CharField(
        widget=forms.Textarea(attrs={'class': 'form-control', 'rows': 5})
    )
    rating = forms.IntegerField(
        min_value=1,
        max_value=10,
        widget=forms.NumberInput(attrs={'class': 'form-control'})
    )
```

### ModelForm (рекомендуется)
```python
class MovieModelForm(forms.ModelForm):
    class Meta:
        model = Movie
        fields = ['title', 'description', 'rating', 'poster', 'category']
        widgets = {
            'title': forms.TextInput(attrs={'class': 'form-control'}),
            'description': forms.Textarea(attrs={'class': 'form-control', 'rows': 5}),
        }

class ReviewForm(forms.ModelForm):
    class Meta:
        model = Review
        fields = ['rating', 'comment']
```

### Обработка формы в представлении
```python
from django.shortcuts import render, redirect
from .forms import MovieModelForm

def create_movie(request):
    if request.method == 'POST':
        form = MovieModelForm(request.POST, request.FILES)
        if form.is_valid():
            form.save()
            return redirect('catalog:movie_list')
    else:
        form = MovieModelForm()
    
    return render(request, 'movie_form.html', {'form': form})
```

### Шаблон формы
```html
<form method="post" enctype="multipart/form-data">
    {% csrf_token %}
    
    {% for field in form %}
        <div class="form-group">
            {{ field.label_tag }}
            {{ field }}
            {% if field.errors %}
                <ul class="errors">
                {% for error in field.errors %}
                    <li>{{ error }}</li>
                {% endfor %}
                </ul>
            {% endif %}
        </div>
    {% endfor %}
    
    <button type="submit" class="btn btn-primary">Сохранить</button>
</form>
```

**Важно:** Всегда используйте `{% csrf_token %}` для защиты от CSRF атак!

## 7. Аутентификация и авторизация

### Встроенная система аутентификации Django
```python
# views.py
from django.contrib.auth import authenticate, login, logout
from django.contrib.auth.models import User
from django.contrib.auth.decorators import login_required
from django.views.generic import View

class RegisterView(View):
    def post(self, request):
        username = request.POST['username']
        email = request.POST['email']
        password = request.POST['password']
        
        if User.objects.filter(username=username).exists():
            return render(request, 'register.html', {'error': 'Username уже занят'})
        
        user = User.objects.create_user(username, email, password)
        login(request, user)
        return redirect('home')

def login_view(request):
    if request.method == 'POST':
        username = request.POST['username']
        password = request.POST['password']
        user = authenticate(request, username=username, password=password)
        
        if user is not None:
            login(request, user)
            return redirect('home')
        else:
            return render(request, 'login.html', {'error': 'Неверные данные'})
    
    return render(request, 'login.html')

def logout_view(request):
    logout(request)
    return redirect('home')
```

### Проверка прав доступа
```python
from django.contrib.auth.decorators import login_required, permission_required

@login_required(login_url='login')
def edit_movie(request, pk):
    movie = get_object_or_404(Movie, pk=pk)
    
    if request.user != movie.author and not request.user.is_staff:
        return HttpResponseForbidden('У вас нет прав')
    
    # ... код редактирования ...

@permission_required('catalog.change_movie')
def approve_movie(request, pk):
    movie = get_object_or_404(Movie, pk=pk)
    movie.status = 'published'
    movie.save()
    return redirect('movie_detail', slug=movie.slug)
```

### В шаблоне
```html
{% if user.is_authenticated %}
    <p>Добро пожаловать, {{ user.username }}!</p>
    <a href="{% url 'logout' %}">Выход</a>
{% else %}
    <a href="{% url 'login' %}">Вход</a>
    <a href="{% url 'register' %}">Регистрация</a>
{% endif %}

{% if user.is_superuser %}
    <a href="/admin/">Админ панель</a>
{% endif %}
```

## 8. Миграции и управление БД

### Основные команды миграций
```bash
# Создать миграцию на основе изменений моделей
python manage.py makemigrations

# Указать конкретное приложение
python manage.py makemigrations catalog

# Применить миграции к БД
python manage.py migrate

# Показать статус миграций
python manage.py showmigrations

# Откатить последнюю миграцию
python manage.py migrate app_name 0001

# Изучить SQL миграции без применения
python manage.py sqlmigrate catalog 0001
```

### Пример миграции
```python
# migrations/0001_initial.py
from django.db import migrations, models
import django.db.models.deletion

class Migration(migrations.Migration):
    initial = True
    
    dependencies = []
    
    operations = [
        migrations.CreateModel(
            name='Category',
            fields=[
                ('id', models.AutoField(primary_key=True)),
                ('name', models.CharField(max_length=100, unique=True)),
                ('created_at', models.DateTimeField(auto_now_add=True)),
            ],
        ),
        migrations.CreateModel(
            name='Movie',
            fields=[
                ('id', models.AutoField(primary_key=True)),
                ('title', models.CharField(max_length=200)),
                ('category', models.ForeignKey(on_delete=django.db.models.deletion.CASCADE, to='catalog.category')),
            ],
        ),
    ]
```

### Best Practices для миграций
1. **Мигрируйте всегда**, прежде чем коммитить код
2. **Даже пустые миграции** важны - сохраняют историю
3. **Не редактируйте применённые миграции** - создавайте новые
4. **Используйте `--empty`** для создания пустых миграций с данными

```bash
# Создать пустую миграцию для изменения данных
python manage.py makemigrations catalog --empty --name add_default_category
```

## 9. REST API с Django REST Framework

### Установка
```bash
pip install djangorestframework
```

Добавить в `INSTALLED_APPS`:
```python
INSTALLED_APPS = [
    # ...
    'rest_framework',
]
```

### Создание Serializers
```python
# serializers.py
from rest_framework import serializers
from .models import Movie, Category

class CategorySerializer(serializers.ModelSerializer):
    class Meta:
        model = Category
        fields = ('id', 'name')

class MovieSerializer(serializers.ModelSerializer):
    category = CategorySerializer(read_only=True)
    
    class Meta:
        model = Movie
        fields = ('id', 'title', 'description', 'rating', 'poster', 'category')
```

### ViewSets и Routers
```python
# views.py
from rest_framework import viewsets
from .serializers import MovieSerializer, CategorySerializer
from .models import Movie, Category

class MovieViewSet(viewsets.ModelViewSet):
    queryset = Movie.objects.all()
    serializer_class = MovieSerializer
    lookup_field = 'slug'
    
    def get_queryset(self):
        return Movie.objects.filter(is_active=True)

class CategoryViewSet(viewsets.ReadOnlyModelViewSet):
    queryset = Category.objects.all()
    serializer_class = CategorySerializer
```

### URL маршруты
```python
# urls.py
from django.urls import path, include
from rest_framework.routers import DefaultRouter
from . import views

router = DefaultRouter()
router.register('movies', views.MovieViewSet)
router.register('categories', views.CategoryViewSet)

urlpatterns = [
    path('api/', include(router.urls)),
    path('api-auth/', include('rest_framework.urls')),
]
```

### Результаты API
```
GET /api/movies/
GET /api/movies/{id}/
GET /api/movies/?search=название
POST /api/movies/ (создание)
PUT /api/movies/{id}/ (обновление)
DELETE /api/movies/{id}/
```

## 10. Тестирование приложения

### Unit-тесты моделей
```python
# tests.py
from django.test import TestCase
from .models import Movie, Category

class MovieModelTest(TestCase):
    def setUp(self):
        self.category = Category.objects.create(name='Action')
        self.movie = Movie.objects.create(
            title='Test Movie',
            slug='test-movie',
            category=self.category,
            duration=120,
            rating=8
        )
    
    def test_movie_creation(self):
        self.assertEqual(self.movie.title, 'Test Movie')
        self.assertEqual(self.movie.category.name, 'Action')
    
    def test_movie_str(self):
        self.assertEqual(str(self.movie), 'Test Movie')
```

### Тесты views
```python
class MovieListViewTest(TestCase):
    def setUp(self):
        self.category = Category.objects.create(name='Drama')
        Movie.objects.create(
            title='Movie 1',
            slug='movie-1',
            category=self.category,
            is_active=True,
            duration=100,
            rating=7
        )
    
    def test_movie_list_view(self):
        response = self.client.get('/movies/')
        self.assertEqual(response.status_code, 200)
        self.assertContains(response, 'Movie 1')
    
    def test_movie_detail_view(self):
        response = self.client.get('/movies/movie-1/')
        self.assertEqual(response.status_code, 200)
        self.assertEqual(response.context['movie'].title, 'Movie 1')
```

### Запуск тестов
```bash
# Запустить все тесты проекта
python manage.py test

# Запустить тесты конкретного приложения
python manage.py test catalog

# Запустить конкретный класс тестов
python manage.py test catalog.tests.MovieModelTest

# С покрытием кода (установить coverage)
pip install coverage
coverage run --source='.' manage.py test
coverage report
coverage html
```

## 11. Развертывание на продакшене

### Подготовка settings.py
```python
# settings.py
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()  # Загружаем из .env файла

# SECURITY
DEBUG = os.getenv('DEBUG', 'False') == 'True'
SECRET_KEY = os.getenv('SECRET_KEY')
ALLOWED_HOSTS = os.getenv('ALLOWED_HOSTS', 'localhost').split(',')

# Database
DATABASES = {
    'default': {
        'ENGINE': 'django.db.backends.postgresql',
        'NAME': os.getenv('DB_NAME'),
        'USER': os.getenv('DB_USER'),
        'PASSWORD': os.getenv('DB_PASSWORD'),
        'HOST': os.getenv('DB_HOST', 'localhost'),
        'PORT': os.getenv('DB_PORT', '5432'),
    }
}

# Static files
STATIC_URL = '/static/'
STATIC_ROOT = os.path.join(BASE_DIR, 'staticfiles')
MEDIA_URL = '/media/'
MEDIA_ROOT = os.path.join(BASE_DIR, 'media')

# HTTPS
SECURE_SSL_REDIRECT = not DEBUG
SESSION_COOKIE_SECURE = not DEBUG
CSRF_COOKIE_SECURE = not DEBUG
```

### .env файл (.gitignore обязателен!)
```
DEBUG=False
SECRET_KEY=your-super-secret-key-here
ALLOWED_HOSTS=example.com,www.example.com
DB_ENGINE=django.db.backends.postgresql
DB_NAME=cinema_db
DB_USER=postgres
DB_PASSWORD=your-password
DB_HOST=localhost
DB_PORT=5432
```

### Gunicorn и Nginx
```bash
# Установить Gunicorn
pip install gunicorn

# Запустить через Gunicorn
gunicorn cinema.wsgi:application --bind 0.0.0.0:8000
```

### Nginx конфиг
```nginx
server {
    listen 80;
    server_name example.com www.example.com;
    
    location / {
        proxy_pass http://127.0.0.1:8000;
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
    }
    
    location /static/ {
        alias /path/to/project/staticfiles/;
    }
    
    location /media/ {
        alias /path/to/project/media/;
    }
}
```

### Docker (опционально)
```dockerfile
FROM python:3.9-slim

WORKDIR /app
COPY requirements.txt .
RUN pip install -r requirements.txt

COPY . .
RUN python manage.py collectstatic --noinput

CMD ["gunicorn", "cinema.wsgi:application", "--bind", "0.0.0.0:8000"]
```

### Checklist перед продакшеном
- ✅ `DEBUG = False`
- ✅ `SECRET_KEY` генерируется и хранится в .env
- ✅ `ALLOWED_HOSTS` содержит только ваши домены
- ✅ `python manage.py check --deploy`
- ✅ `collectstatic` для сбора статических файлов
- ✅ HTTPS сертификат (Let's Encrypt)
- ✅ Database backup стратегия
- ✅ Логирование и мониторинг настроены

## Советы и Best Practices

### Организация кода
```
project/
├── apps/                    # Все приложения в одной папке
│   ├── catalog/
│   ├── users/
│   └── api/
├── config/                  # Конфигурация
│   ├── settings/
│   │   ├── base.py
│   │   ├── dev.py
│   │   └── prod.py
│   └── urls.py
├── static/
├── templates/
├── tests/                   # Отдельная папка для тестов
└── manage.py
```

### 10 Золотых Правил Django
1. **DRY (Don't Repeat Yourself)** - не повторяйте код
2. **Используйте ORM Django** вместо raw SQL
3. **Используйте миграции** - всегда!
4. **Пишите тесты** - покрытие минимум 80%
5. **Используйте virtualenv/venv** - всегда изолируйте зависимости
6. **Используйте .env файл** для чувствительных данных
7. **Кэшируйте** - используйте Redis для часто используемых данных
8. **Логируйте** - сохраняйте логи для отладки
9. **Используйте signals аккуратно** - они могут замедлить приложение
10. **Профилируйте** - используйте Django Debug Toolbar

### Неправильно ❌
```python
# N+1 проблема
movies = Movie.objects.all()
for movie in movies:
    print(movie.category.name)  # Запрос к БД на каждой итерации

# Небезопасные запросы
query = "SELECT * FROM movies WHERE id = " + str(request.GET['id'])
# SQL Injection!
```

### Правильно ✅
```python
# Используйте select_related
movies = Movie.objects.select_related('category').all()
for movie in movies:
    print(movie.category.name)  # Нет дополнительных запросов

# Используйте ORM
movie = Movie.objects.filter(id=request.GET.get('id')).first()
```

### Полезные пакеты
```bash
pip install django-extensions      # Полезные команды
pip install django-debug-toolbar   # Отладка во время разработки
pip install celery                 # Асинхронные задачи
pip install redis                  # Кэширование
pip install black                  # Форматирование кода
pip install flake8                 # Линтинг
```

### Структура requirements.txt
```
Django==4.2.0
djangorestframework==3.14.0
python-dotenv==1.0.0
Pillow==10.0.0
redis==5.0.0
celery==5.3.0
psycopg2-binary==2.9.0
gunicorn==21.0.0
black==23.7.0
flake8==6.0.0
```

### Git Workflow
```bash
# Создать новую ветку для features
git checkout -b feature/new-feature

# Add, commit, push
git add .
git commit -m "feat: описание изменений"
git push origin feature/new-feature

# Создать Pull Request на GitHub
# После ревью и одобрения - merge
```

### Команды которые должны быть в памяти
```bash
python manage.py shell              # Интерактивная консоль Django
python manage.py dbshell            # Консоль БД
python manage.py runserver          # Запуск сервера
python manage.py makemigrations     # Создать миграции
python manage.py migrate            # Применить миграции
python manage.py createsuperuser    # Создать админа
python manage.py collectstatic      # Собрать статические файлы
```

---

## Ресурсы для дальнейшего обучения

- **Официальная документация:** https://docs.djangoproject.com/
- **Django Best Practices:** Two Scoops of Django
- **Real Python Django tutorials:** https://realpython.com/tutorials/django/
- **Django Discord Community:** https://discord.gg/django

**Happy Coding! 🚀**